# 03 Train Image Model

Train EfficientNet-B0 with transfer learning for binary deepfake image classification.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

DATA_ROOT = Path("../../data/splits/images")
MODEL_DIR = Path("../../models/image")
MODEL_PATH = MODEL_DIR / "best_model.pth"
SPLITS = ["train", "val", "test"]
CLASSES = ["real", "fake"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def collect_image_paths(data_root: Path) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        for class_name in CLASSES:
            class_dir = data_root / split / class_name
            print(f"Scanning {class_dir} | exists={class_dir.exists()}")
            if not class_dir.exists():
                continue
            for path in sorted(class_dir.iterdir()):
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({"path": str(path), "split": split, "label": class_name, "label_id": 1 if class_name == "fake" else 0})
    return pd.DataFrame(rows)


df = collect_image_paths(DATA_ROOT)
print(f"Loaded records: {len(df)}")
display(df.head())
display(df.groupby(["split", "label"]).size().unstack(fill_value=0))

Scanning ..\data\splits\images\train\real | exists=True
Scanning ..\data\splits\images\train\fake | exists=True
Scanning ..\data\splits\images\val\real | exists=True
Scanning ..\data\splits\images\val\fake | exists=True
Scanning ..\data\splits\images\test\real | exists=True
Scanning ..\data\splits\images\test\fake | exists=True
Loaded records: 57098


,path,split,label,label_id
0,..\data\splits\images\train\real\real_100.jpg,train,real,0
1,..\data\splits\images\train\real\real_10001.jpg,train,real,0
2,..\data\splits\images\train\real\real_10002.jpg,train,real,0
3,..\data\splits\images\train\real\real_10006.jpg,train,real,0
4,..\data\splits\images\train\real\real_10013.jpg,train,real,0


label,fake,real
split,,
test,1647,1623
train,21000,21000
val,5892,5936


In [3]:
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class DeepfakeImageDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        print(f"Dataset initialized with {len(self.dataframe)} images")

    def __len__(self):
        return len(self.dataframe)

    def _load_image(self, path: str):
        try:
            with Image.open(path) as image:
                return image.convert("RGB")
        except (OSError, UnidentifiedImageError) as error:
            print(f"Warning: corrupted image replaced with blank tensor: {path} | {error}")
            return Image.new("RGB", (224, 224), color=(0, 0, 0))

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = self._load_image(row["path"])
        label = torch.tensor(row["label_id"], dtype=torch.float32)
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [4]:
train_dataset = DeepfakeImageDataset(df[df["split"] == "train"], transform=train_transform)
val_dataset = DeepfakeImageDataset(df[df["split"] == "val"], transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")
images, labels = next(iter(train_loader))
print(f"Debug train batch images: {images.shape}")
print(f"Debug train batch labels: {labels.shape}")
print(f"Debug train labels: {labels[:10].tolist()}")

Dataset initialized with 42000 images
Dataset initialized with 11828 images
Train size: 42000 | Val size: 11828
Debug train batch images: torch.Size([32, 3, 224, 224])
Debug train batch labels: torch.Size([32])
Debug train labels: [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0]


In [5]:
def build_model() -> nn.Module:
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)
    return model


model = build_model().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model.classifier)
print(f"Model output units: {model.classifier[1].out_features}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Asus/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:05<00:00, 3.89MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)
Model output units: 1


In [6]:
def run_epoch(model, loader, criterion, optimizer=None, phase="train"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for batch_idx, (images, labels) in enumerate(loader):
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            if batch_idx == 0:
                print(f"{phase} batch images shape: {images.shape}")
                print(f"{phase} batch labels shape: {labels.shape}")

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= 0.5).float()

            if batch_idx == 0:
                print(f"{phase} logits: {logits[:5].detach().cpu().view(-1).tolist()}")
                print(f"{phase} probabilities: {probabilities[:5].detach().cpu().view(-1).tolist()}")
                print(f"{phase} predictions: {predictions[:5].detach().cpu().view(-1).tolist()}")

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            correct += (predictions == labels).sum().item()
            total += batch_size

    epoch_loss = running_loss / max(total, 1)
    epoch_accuracy = correct / max(total, 1)
    return epoch_loss, epoch_accuracy

In [7]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer, phase="train")
    val_loss, val_accuracy = run_epoch(model, val_loader, criterion, optimizer=None, phase="val")

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
    )

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, train_acc={train_accuracy:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_accuracy:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Saved best model to {MODEL_PATH} with val_loss={best_val_loss:.4f}")

history_df = pd.DataFrame(history)
display(history_df)


Epoch 1/5
train batch images shape: torch.Size([32, 3, 224, 224])
train batch labels shape: torch.Size([32, 1])
train logits: [0.2623059153556824, 0.11098689585924149, 0.011800142005085945, 0.04414676874876022, 0.032201558351516724]
train probabilities: [0.565203070640564, 0.5277182459831238, 0.5029500126838684, 0.5110349059104919, 0.5080496668815613]
train predictions: [1.0, 1.0, 1.0, 1.0, 1.0]
val batch images shape: torch.Size([32, 3, 224, 224])
val batch labels shape: torch.Size([32, 1])
val logits: [-5.160943984985352, -9.312813758850098, -5.904494285583496, -9.67681884765625, -4.255709648132324]
val probabilities: [0.005703565198928118, 9.025206963997334e-05, 0.002719743177294731, 6.271678284974769e-05, 0.013984677381813526]
val predictions: [0.0, 0.0, 0.0, 0.0, 0.0]
Epoch 1: train_loss=0.1156, train_acc=0.9550, val_loss=0.0979, val_acc=0.9618
Saved best model to ..\models\image\best_model.pth with val_loss=0.0979

Epoch 2/5
train batch images shape: torch.Size([32, 3, 224, 224]

,epoch,train_loss,train_accuracy,val_loss,val_accuracy
0,1,0.115600,0.955000,0.097924,0.961786
1,2,0.046899,0.982381,0.093442,0.969395
2,3,0.032584,0.987429,0.077068,0.975905
3,4,0.025938,0.989238,0.080825,0.974467
4,5,0.020759,0.991476,0.077336,0.977088
